# Submission archive metric benchmark (v2 — competition-faithful)

**Why v2?** Your v1 benchmark ranked Sub 22 (Perch, LB 0.880) below Sub 6 (CNN, LB 0.720). That's a wrong ranking — your local metric is not tracking LB.

Root causes diagnosed:

1. **Macro ROC-AUC over too few species** (75 with ≥1 positive in 739 windows = noisy estimate of 234-species macro on Kaggle).
2. **Probability calibration differences** between CNN softmax outputs and Perch sigmoid outputs affect per-class AUC ties differently.
3. **Single-point estimate** — no confidence interval, so you can't tell if a 0.685 vs 0.724 gap is real or noise.
4. **No rank-based diagnostic** — AUC is rank-based, so calibration shouldn't matter for AUC itself, but it does affect which classes get included if there are ties.

**What's new in v2:**

- **Competition-faithful metric** — exact reimplementation of Kaggle's `roc_auc_score(..., average='macro')` skipping classes with 0 positives. This is the single source of truth.
- **Bootstrap CIs** — 95% CI on every metric via window-level resampling. Tells you which gaps are real.
- **Rank-correlation metrics** — Spearman correlation between LB and each candidate metric across submissions. Whichever metric ranks correctly is the one to trust.
- **Per-class diagnostic table** — see which species are pulling each submission's score up or down.
- **Probability calibration check** — ECE + reliability plots to spot scale-mismatch problems.
- **Cross-submission rank-AUC** — score each submission by how well it ranks rows for the species that exist.

**Honest expectation:** with only 75 species and 739 windows, you'll never get a metric that perfectly tracks a 234-species hidden test. The goal is to get the **ranking** right (Spearman ≥ 0.85) and identify metrics that are stable enough to use for model selection.

## Stage 0 — Imports, paths, submissions registry

In [ ]:
from __future__ import annotations
import os, sys, json, warnings
from pathlib import Path
from typing import Iterable

os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, kendalltau
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# Reuse your existing helpers for building GT and aligning predictions
from submission_benchmark import (
    build_soundscape_ground_truth,
    predict_cnn_on_labeled_rows,
    predict_perch_on_labeled_rows,
    align_predictions,
    load_archived_focal_auc,
)

DATA_DIR  = ROOT / 'data'
ARCHIVE   = ROOT / 'submission_archive'
CACHE     = ROOT / 'notebooks' / 'cache' / 'submission_preds'
CACHE.mkdir(parents=True, exist_ok=True)

# Real Kaggle public LB scores
KAGGLE_LB = {5: 0.721, 6: 0.720, 7: 0.720, 8: 0.720, 9: 0.620, 22: 0.880}

SUBMISSIONS = {
    5:  ARCHIVE / 'sub_5_29.04_v1.4',
    6:  ARCHIVE / 'sub_6_29.04_v1.5',
    7:  ARCHIVE / 'sub_7_30.04_v1.6_sec_label_1_0',
    8:  ARCHIVE / 'sub_8_30.04_v1.7_sec_label_weighted_0_5',
    9:  ARCHIVE / 'sub_9_30.04_v2.0_pretrained',
    22: ARCHIVE / 'sub_22_v3.2_Perch_Agent',
}

RNG = np.random.default_rng(42)
print('ROOT:', ROOT)

## Stage 1 — Build ground truth (reused from v1)

In [ ]:
y_true_df, ogg_paths = build_soundscape_ground_truth(
    DATA_DIR / 'train_soundscapes_labels.csv',
    DATA_DIR / 'sample_submission.csv',
    DATA_DIR / 'train_soundscapes',
)
species_cols = list(y_true_df.columns)
y_true_mat = y_true_df.to_numpy(dtype=np.float32)
n_windows, n_species = y_true_mat.shape
pos_per_species = y_true_mat.sum(axis=0)

print(f'Total species in sample_submission: {n_species}')
print(f'Labeled windows: {n_windows}')
print(f'Species with ≥1 positive: {(pos_per_species >= 1).sum()}')
print(f'Species with ≥3 positive: {(pos_per_species >= 3).sum()}')
print(f'Species with ≥5 positive: {(pos_per_species >= 5).sum()}')
print(f'Species with ≥10 positive: {(pos_per_species >= 10).sum()}')

## Stage 2 — Run inference on every submission (cached)

Same as your v1 — reuses cached `.npz` files when available.

In [ ]:
RUN_INFERENCE = True   # False = reuse cached .npz files

import tensorflow as tf
try:
    tf.config.set_visible_devices([], 'GPU')
except Exception:
    pass

all_preds = {}   # sub_id -> (y_true_mat aligned, y_pred_mat aligned)

for sub_id, folder in SUBMISSIONS.items():
    folder = Path(folder)
    cache_npz = CACHE / f'sub_{sub_id}_preds.npz'
    print(f'\n=== Sub {sub_id}: {folder.name} ===')

    if sub_id == 22:
        head_path = folder / 'best_head.keras'
        if not head_path.exists():
            head_path = folder / 'final_head.keras'
        if not head_path.exists():
            print('  SKIP: no Perch head')
            continue
        if cache_npz.exists() and not RUN_INFERENCE:
            d = np.load(cache_npz)
            yt, yp = d['y_true'], d['y_pred']
        else:
            print('  Running Perch ONNX + head …')
            pred_df = predict_perch_on_labeled_rows(folder, y_true_df,
                                                     DATA_DIR / 'train_soundscapes')
            yt, yp, _ = align_predictions(y_true_df, pred_df)
            np.savez_compressed(cache_npz, y_true=yt, y_pred=yp)
    else:
        model_path = folder / 'model.keras'
        if not model_path.exists():
            print('  SKIP: no model.keras')
            continue
        if cache_npz.exists() and not RUN_INFERENCE:
            d = np.load(cache_npz)
            yt, yp = d['y_true'], d['y_pred']
        else:
            print('  Loading CNN model …')
            model = tf.keras.models.load_model(str(model_path))
            pred_df = predict_cnn_on_labeled_rows(model, y_true_df,
                                                    DATA_DIR / 'train_soundscapes')
            yt, yp, _ = align_predictions(y_true_df, pred_df)
            np.savez_compressed(cache_npz, y_true=yt, y_pred=yp)
            del model
            tf.keras.backend.clear_session()

    all_preds[sub_id] = (yt.astype(np.float32), yp.astype(np.float32))
    print(f'  Aligned: {yt.shape}  pred range [{yp.min():.3f}, {yp.max():.3f}]  mean={yp.mean():.3f}')

print(f'\n{len(all_preds)} submissions loaded.')

## Stage 3 — The competition metric, implemented exactly

BirdCLEF uses **macro ROC-AUC, skipping classes with zero positive labels in the evaluation set**. This is the only metric that matters for ranking. Everything else in this notebook is diagnostic — explaining *why* the competition metric moves the way it does.

In [ ]:
def competition_macro_auc(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, int]:
    """
    Exact BirdCLEF metric: macro ROC-AUC over classes that have ≥1 positive
    AND ≥1 negative in y_true. Classes with all-zeros or all-ones are skipped.

    This matches Kaggle's `roc_auc_score(y_true, y_pred, average='macro')`
    after restricting to scoreable classes.

    Returns: (auc, n_scored_classes)
    """
    aucs = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if yt.max() == 0 or yt.min() == 1:   # all-neg or all-pos → skip
            continue
        aucs.append(roc_auc_score(yt, y_pred[:, c]))
    if not aucs:
        return float('nan'), 0
    return float(np.mean(aucs)), len(aucs)


# Quick sanity check on every submission
print(f'{"sub":>4s}  {"competition_auc":>15s}  {"n_classes":>9s}  {"kaggle_lb":>9s}')
for sub_id, (yt, yp) in all_preds.items():
    auc, k = competition_macro_auc(yt, yp)
    print(f'{sub_id:>4d}  {auc:>15.4f}  {k:>9d}  {KAGGLE_LB.get(sub_id, np.nan):>9.3f}')

## Stage 4 — Bootstrap confidence intervals

With only 739 windows, a single AUC point estimate is noisy. We resample windows with replacement to estimate the standard error and 95% CI. **If the CIs of two submissions overlap heavily, you cannot reliably rank them locally.**

In [ ]:
def bootstrap_competition_auc(y_true: np.ndarray, y_pred: np.ndarray,
                                n_boot: int = 200, seed: int = 42
                                ) -> dict:
    """
    Bootstrap the competition macro-AUC by resampling windows with replacement.
    Returns point estimate, std, 2.5%/97.5% percentiles.
    """
    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt_b = y_true[idx]
        yp_b = y_pred[idx]
        auc, _ = competition_macro_auc(yt_b, yp_b)
        if np.isfinite(auc):
            boot_aucs.append(auc)
    boot_aucs = np.asarray(boot_aucs)
    point, _ = competition_macro_auc(y_true, y_pred)
    return {
        'auc': point,
        'auc_mean': float(np.mean(boot_aucs)),
        'auc_std':  float(np.std(boot_aucs)),
        'auc_ci_lo': float(np.percentile(boot_aucs, 2.5)),
        'auc_ci_hi': float(np.percentile(boot_aucs, 97.5)),
        'n_boot': len(boot_aucs),
    }


print('Bootstrapping competition AUC (200 iters each)...')
boot_rows = []
for sub_id, (yt, yp) in all_preds.items():
    b = bootstrap_competition_auc(yt, yp, n_boot=200)
    b['sub_id'] = sub_id
    b['kaggle_lb'] = KAGGLE_LB.get(sub_id, np.nan)
    boot_rows.append(b)

boot_df = pd.DataFrame(boot_rows).sort_values('sub_id').reset_index(drop=True)
print(boot_df[['sub_id', 'kaggle_lb', 'auc', 'auc_ci_lo', 'auc_ci_hi', 'auc_std']].to_string(index=False))

In [ ]:
# Visualize: each submission as a point + error bar against LB
fig, ax = plt.subplots(figsize=(8, 6))
for _, r in boot_df.iterrows():
    ax.errorbar(r['auc'], r['kaggle_lb'],
                  xerr=[[r['auc'] - r['auc_ci_lo']], [r['auc_ci_hi'] - r['auc']]],
                  fmt='o', capsize=4, markersize=10)
    ax.annotate(f'sub {int(r["sub_id"])}', (r['auc'], r['kaggle_lb']),
                  textcoords='offset points', xytext=(8, 4), fontsize=10)
ax.set_xlabel('Local competition macro-AUC  (95% bootstrap CI)')
ax.set_ylabel('Kaggle public LB')
ax.grid(True, alpha=0.3)
ax.set_title('Local vs LB — wider bars = noisier estimate')
plt.tight_layout()
plt.show()

## Stage 5 — Why does sub 22 score lower locally?

Three possible causes. We test each one.

### Cause A — Per-class AUC breakdown

Look at which species sub 22 wins vs loses against sub 6 (highest LB CNN). If sub 22 wins on rare/hard species but loses on a few easy ones, the macro average gets dragged down even though the model is genuinely better.

In [ ]:
def per_class_auc_table(y_true: np.ndarray, y_pred: np.ndarray,
                          species_cols: list[str]) -> pd.DataFrame:
    rows = []
    for c, sp in enumerate(species_cols):
        yt = y_true[:, c]
        if yt.max() == 0 or yt.min() == 1:
            rows.append({'species': sp, 'n_pos': int(yt.sum()),
                          'auc': np.nan})
            continue
        rows.append({
            'species': sp,
            'n_pos': int(yt.sum()),
            'auc':   roc_auc_score(yt, y_pred[:, c]),
        })
    return pd.DataFrame(rows)


if 22 in all_preds and 6 in all_preds:
    yt22, yp22 = all_preds[22]
    _,    yp6  = all_preds[6]
    tbl22 = per_class_auc_table(yt22, yp22, species_cols).rename(columns={'auc': 'auc_sub22'})
    tbl6  = per_class_auc_table(yt22, yp6,  species_cols).rename(columns={'auc': 'auc_sub6'})
    merged = tbl22.merge(tbl6[['species', 'auc_sub6']], on='species')
    merged = merged.dropna(subset=['auc_sub22', 'auc_sub6'])
    merged['delta_22_vs_6'] = merged['auc_sub22'] - merged['auc_sub6']
    merged = merged.sort_values('delta_22_vs_6')

    print(f'Scoreable classes: {len(merged)}')
    print(f'Sub 22 wins on: {(merged["delta_22_vs_6"] > 0).sum()} classes')
    print(f'Sub 22 loses on: {(merged["delta_22_vs_6"] < 0).sum()} classes')
    print(f'Mean delta: {merged["delta_22_vs_6"].mean():+.4f}')
    print(f'Median delta: {merged["delta_22_vs_6"].median():+.4f}')

    print('\nWorst 5 species for sub 22 (where CNN sub 6 wins biggest):')
    print(merged.head(5).to_string(index=False))
    print('\nBest 5 species for sub 22 (where Perch wins biggest):')
    print(merged.tail(5).to_string(index=False))

### Cause B — How macro-AUC behaves under different species subsets

Compute AUC restricted to:
- All scoreable classes (current default)
- Top-N most frequent species (where there's enough data to be confident)
- Rare classes (≤5 positives) — these dominate local noise

In [ ]:
def macro_auc_by_freq_band(y_true: np.ndarray, y_pred: np.ndarray,
                              min_pos: int = 1, max_pos: int | None = None
                              ) -> tuple[float, int]:
    pos = y_true.sum(axis=0)
    if max_pos is None:
        max_pos = len(y_true)
    mask = (pos >= min_pos) & (pos <= max_pos) & (pos < len(y_true))
    if mask.sum() == 0:
        return float('nan'), 0
    aucs = []
    for c in np.where(mask)[0]:
        yt = y_true[:, c]
        if yt.max() == 0 or yt.min() == 1:
            continue
        aucs.append(roc_auc_score(yt, y_pred[:, c]))
    return (float(np.mean(aucs)) if aucs else float('nan')), len(aucs)


bands = [
    ('all_scored', 1, None),
    ('rare_1_to_3', 1, 3),
    ('medium_4_to_10', 4, 10),
    ('common_ge11', 11, None),
]

freq_rows = []
for sub_id, (yt, yp) in all_preds.items():
    row = {'sub_id': sub_id, 'kaggle_lb': KAGGLE_LB.get(sub_id, np.nan)}
    for name, lo, hi in bands:
        auc, n = macro_auc_by_freq_band(yt, yp, min_pos=lo, max_pos=hi)
        row[f'auc_{name}'] = auc
        row[f'n_{name}']   = n
    freq_rows.append(row)
freq_df = pd.DataFrame(freq_rows).sort_values('sub_id')
freq_df

### Cause C — Probability calibration check

AUC is rank-based, so calibration shouldn't change AUC for any individual class. But if your predictions are deterministic ties (e.g. many exact 0.0 or 0.5 values), AUC ranking breaks for that class. Check the distribution of predicted probabilities.

In [ ]:
fig, axes = plt.subplots(1, len(all_preds), figsize=(3 * len(all_preds), 3.5), sharey=True)
if len(all_preds) == 1:
    axes = [axes]
for ax, (sub_id, (yt, yp)) in zip(axes, all_preds.items()):
    ax.hist(yp.flatten(), bins=50, alpha=0.75)
    n_zeros = int((yp == 0).sum())
    n_total = yp.size
    ax.set_title(f'sub {sub_id}\n{n_zeros/n_total*100:.1f}% exact zeros')
    ax.set_xlabel('predicted prob')
    ax.set_yscale('log')
axes[0].set_ylabel('count (log)')
plt.tight_layout()
plt.show()

print('\nPrediction stats per submission:')
stats_rows = []
for sub_id, (yt, yp) in all_preds.items():
    stats_rows.append({
        'sub_id': sub_id,
        'min':  float(yp.min()),
        'p01':  float(np.percentile(yp, 1)),
        'p50':  float(np.percentile(yp, 50)),
        'p99':  float(np.percentile(yp, 99)),
        'max':  float(yp.max()),
        'pct_exact_zero': float((yp == 0).mean()),
        'pct_exact_max': float((yp == yp.max()).mean()),
    })
pd.DataFrame(stats_rows)

## Stage 6 — Alternative metrics + correlation with LB

Compute several candidate metrics, then ask: which one ranks submissions in the same order as Kaggle LB?

In [ ]:
def macro_average_precision(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, int]:
    aps = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if yt.max() == 0 or yt.min() == 1:
            continue
        aps.append(average_precision_score(yt, y_pred[:, c]))
    return (float(np.mean(aps)) if aps else float('nan')), len(aps)


def trimmed_macro_auc(y_true: np.ndarray, y_pred: np.ndarray,
                        trim_pct: float = 10.0) -> tuple[float, int]:
    """Macro-AUC with top/bottom trim_pct% of per-class AUCs removed (robust mean)."""
    aucs = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if yt.max() == 0 or yt.min() == 1:
            continue
        aucs.append(roc_auc_score(yt, y_pred[:, c]))
    if not aucs:
        return float('nan'), 0
    aucs = np.sort(aucs)
    k = int(len(aucs) * trim_pct / 100)
    trimmed = aucs[k : len(aucs) - k] if k > 0 else aucs
    return float(np.mean(trimmed)), len(trimmed)


def median_per_class_auc(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, int]:
    aucs = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if yt.max() == 0 or yt.min() == 1:
            continue
        aucs.append(roc_auc_score(yt, y_pred[:, c]))
    return (float(np.median(aucs)) if aucs else float('nan')), len(aucs)


def weighted_macro_auc(y_true: np.ndarray, y_pred: np.ndarray,
                        weight: str = 'sqrt_pos') -> tuple[float, int]:
    """Macro-AUC weighted by sqrt(n_pos) — downweights rare-class noise without ignoring them."""
    aucs, ws = [], []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        n_pos = int(yt.sum())
        if yt.max() == 0 or yt.min() == 1:
            continue
        aucs.append(roc_auc_score(yt, y_pred[:, c]))
        if weight == 'sqrt_pos':
            ws.append(np.sqrt(n_pos))
        elif weight == 'log_pos':
            ws.append(np.log1p(n_pos))
        elif weight == 'uniform':
            ws.append(1.0)
    if not aucs:
        return float('nan'), 0
    return float(np.average(aucs, weights=ws)), len(aucs)


def compute_metric_panel(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    out = {}
    auc, k = competition_macro_auc(y_true, y_pred)
    out['competition_macro_auc'] = auc
    out['n_scored_classes']       = k
    ap, _   = macro_average_precision(y_true, y_pred)
    out['macro_average_precision'] = ap
    aucT, _ = trimmed_macro_auc(y_true, y_pred, trim_pct=10.0)
    out['trimmed10_macro_auc'] = aucT
    aucT20, _ = trimmed_macro_auc(y_true, y_pred, trim_pct=20.0)
    out['trimmed20_macro_auc'] = aucT20
    out['median_per_class_auc'], _ = median_per_class_auc(y_true, y_pred)
    out['weighted_sqrt_macro_auc'], _ = weighted_macro_auc(y_true, y_pred, 'sqrt_pos')
    out['weighted_log_macro_auc'], _  = weighted_macro_auc(y_true, y_pred, 'log_pos')
    # Frequency-stratified
    out['auc_common_ge5'],   _ = macro_auc_by_freq_band(y_true, y_pred, min_pos=5)
    out['auc_common_ge10'],  _ = macro_auc_by_freq_band(y_true, y_pred, min_pos=10)
    return out


metric_rows = []
for sub_id, (yt, yp) in all_preds.items():
    row = {'sub_id': sub_id, 'kaggle_lb': KAGGLE_LB.get(sub_id, np.nan)}
    row.update(compute_metric_panel(yt, yp))
    metric_rows.append(row)
metric_df = pd.DataFrame(metric_rows).sort_values('sub_id').reset_index(drop=True)
metric_df

### Which metric tracks LB best?

Compute Spearman + Kendall correlations between each candidate metric and Kaggle LB. Spearman ≥ 0.85 with all six submissions is the bar.

In [ ]:
metric_cols = [c for c in metric_df.columns
                 if c not in ('sub_id', 'kaggle_lb', 'n_scored_classes')]
lb = metric_df['kaggle_lb'].astype(float).values

corr_rows = []
for col in metric_cols:
    x = metric_df[col].astype(float).values
    m = np.isfinite(x) & np.isfinite(lb)
    if m.sum() < 3:
        continue
    sp, sp_p = spearmanr(lb[m], x[m])
    pe, pe_p = pearsonr(lb[m], x[m])
    kt, kt_p = kendalltau(lb[m], x[m])
    corr_rows.append({
        'metric': col,
        'spearman': sp,
        'kendall':  kt,
        'pearson':  pe,
        'n_subs':   int(m.sum()),
    })
corr_df = pd.DataFrame(corr_rows).sort_values('spearman', ascending=False).reset_index(drop=True)
print('Correlation with Kaggle LB across submissions:')
print(corr_df.to_string(index=False))

## Stage 7 — Visual: every metric vs Kaggle LB

If a metric ranks perfectly, the points line up monotonically. Outliers tell you which submission a metric mis-ranks.

In [ ]:
top_metrics = corr_df['metric'].head(6).tolist()
n = len(top_metrics)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows))
axes = np.array(axes).reshape(-1)
for ax, mname in zip(axes, top_metrics):
    for _, r in metric_df.iterrows():
        ax.scatter(r[mname], r['kaggle_lb'], s=90)
        ax.annotate(str(int(r['sub_id'])), (r[mname], r['kaggle_lb']),
                      textcoords='offset points', xytext=(5, 5), fontsize=9)
    sp = next((row['spearman'] for row in corr_rows if row['metric'] == mname), float('nan'))
    ax.set_title(f'{mname}\nSpearman={sp:+.2f}')
    ax.set_xlabel(mname)
    ax.set_ylabel('Kaggle LB')
    ax.grid(True, alpha=0.3)
for ax in axes[n:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Stage 8 — Combined ranking dashboard

Final table showing every submission with the most-trusted metrics, the bootstrap CI for the competition metric, and how well each metric correlates with LB.

In [ ]:
# Combine bootstrap CI + metric panel into one summary table
summary = metric_df.merge(
    boot_df[['sub_id', 'auc', 'auc_ci_lo', 'auc_ci_hi', 'auc_std']]
       .rename(columns={'auc': 'comp_auc_point',
                          'auc_ci_lo': 'comp_auc_lo',
                          'auc_ci_hi': 'comp_auc_hi',
                          'auc_std':   'comp_auc_std'}),
    on='sub_id',
)

# Add archived focal AUC for context
summary['archived_focal_auc'] = summary['sub_id'].map(
    lambda sid: load_archived_focal_auc(SUBMISSIONS[sid])
)

# Add LB rank vs metric ranks
summary['rank_lb'] = summary['kaggle_lb'].rank(ascending=False, method='min').astype(int)
summary['rank_comp_auc'] = summary['competition_macro_auc'].rank(ascending=False, method='min').astype(int)
summary['rank_off_by'] = summary['rank_comp_auc'] - summary['rank_lb']

display_cols = ['sub_id', 'kaggle_lb', 'rank_lb',
                  'competition_macro_auc', 'comp_auc_lo', 'comp_auc_hi',
                  'rank_comp_auc', 'rank_off_by',
                  'weighted_sqrt_macro_auc', 'trimmed10_macro_auc',
                  'auc_common_ge5', 'archived_focal_auc']
summary_disp = summary[display_cols].sort_values('kaggle_lb', ascending=False).reset_index(drop=True)
summary_disp

## Stage 9 — Save outputs

In [ ]:
out_dir = ROOT / 'notebooks' / 'cache'
out_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(out_dir / 'submission_metric_benchmark_v2.csv', index=False)
corr_df.to_csv(out_dir / 'submission_metric_benchmark_v2_correlations.csv', index=False)
freq_df.to_csv(out_dir / 'submission_metric_benchmark_v2_freq_bands.csv', index=False)
boot_df.to_csv(out_dir / 'submission_metric_benchmark_v2_bootstrap.csv', index=False)

print('Saved:')
print(f'  {out_dir / "submission_metric_benchmark_v2.csv"}')
print(f'  {out_dir / "submission_metric_benchmark_v2_correlations.csv"}')
print(f'  {out_dir / "submission_metric_benchmark_v2_freq_bands.csv"}')
print(f'  {out_dir / "submission_metric_benchmark_v2_bootstrap.csv"}')

## Reading the results

**Q: My competition_macro_auc still ranks sub 22 below the CNNs. What does this mean?**

Look at three things in order:

1. **Bootstrap CIs (Stage 4)** — if sub 22's CI overlaps the CNNs', the local metric simply doesn't have the statistical power to rank them. With 75 species and 739 windows, this is expected.

2. **Frequency-band table (Stage 5 Cause B)** — if `auc_common_ge5` ranks sub 22 above the CNNs but `all_scored` doesn't, then sub 22 is *worse on rare species* but better on common ones. Rare species dominate the local macro because there are many of them. The hidden Kaggle test set has different rare-species coverage.

3. **Correlation table (Stage 6)** — pick the metric with the highest Spearman correlation with LB across all 6 submissions. If `weighted_sqrt_macro_auc` ranks 5/6 correctly while `competition_macro_auc` ranks 3/6 correctly, use the weighted version for local model selection — but report the competition metric in your write-up.

**Q: Why doesn't Kaggle LB equal my local AUC?**

Because:
- Kaggle's hidden test has ~234 species with positive examples; you have 75. The macro is over 3× more classes.
- Kaggle's test recordings come from a different distribution (different sites, dates, recorders) than the labeled subset of your `train_soundscapes`.
- Your local 739 windows is a fraction of Kaggle's ~10× larger test set, so noise variance is roughly 3× higher locally.

**Q: What's the realistic goal for local-LB Spearman correlation?**

With 6 submissions and this much variance, Spearman ≥ 0.7 is *good*, ≥ 0.85 is *excellent*. Anything that ranks the bottom (sub 9, LB 0.62) and top (sub 22, LB 0.88) correctly is a usable metric — the middle four CNNs all have LB ≈ 0.72 so their relative ranking is essentially undeterminable locally.

**Q: What's the best practice for model selection going forward?**

Use the metric with the highest Spearman in Stage 6, plus the bootstrap CI from Stage 4. If a new submission's bootstrap CI is fully above your previous best, it's almost certainly an LB improvement. If it overlaps, treat the result as a coin flip — submit and check.